# Survival TTE Based Evaluations

In [ ]:
# ---- base folder names (define once, reuse below) ----
ckd_event_full     = "365day_future_prediction_outputs_50_full_stage_filter_v8"
ckd_event_subset   = "365day_future_prediction_outputs_50_subset_100_stage_filter_v1"
ckd_patient_full   = "365day_future_prediction_outputs_50_full_stage_filter_patient_level_v2"
ckd_patient_subset = "365day_future_prediction_outputs_50_subset_1000_stage_filter_patient_level_v2"

eskd_event_full   = "365day_future_prediction_outputs_full_stage_filter_eskd_v2"
eskd_event_subset = "365day_future_prediction_outputs_subset_100_stage_filter_eskd_v2"
eskd_patient_full = "365day_future_prediction_outputs_full_stage_filter_eskd_v2_patient_level"
# no subset patient-level eskd data yet -- add eskd_patient_subset here when it exists

# ---- per-model file lists (define once, reuse below) ----
ckd_surv_dirs = [
    "/DeepSurv_LSTM_365DayFutureTarget_detailed_outputs.csv",
    "/DeepSurv_MLP_365DayFutureTarget_detailed_outputs.csv",
    "/DeepSurv_RNN_365DayFutureTarget_detailed_outputs.csv",
    "/DeepSurv_TCN_365DayFutureTarget_detailed_outputs.csv",
    "/DeepSurv_Transformer_365DayFutureTarget_detailed_outputs.csv",
]
eskd_event_dirs   = ["/XGBoost_365DayFuture_Classifier_detailed_outputs_classification.csv"]
eskd_patient_dirs = ["/XGBoost_365DayFuture_Classifier_detailed_outputs_classification_pt_lvl.csv"]

def paths(base, dirs):
    return [f"./{base}{d}" for d in dirs]

# ---- naming convention: <ckd|eskd|ckd_eskd>_<clf|surv>_<event|patient>_<full|subset> ----
presets = {

    # !!
    "ckd_surv_event_full": {
        "fp": f"./{ckd_event_full}", "modifier": "deepsurv",
        "filepaths": paths(ckd_event_full, ckd_surv_dirs),
    },
    "ckd_surv_event_subset": {
        "fp": f"./{ckd_event_subset}", "modifier": "deepsurv",
        "filepaths": paths(ckd_event_subset, ckd_surv_dirs),
    },
    # !!
    "ckd_surv_patient_full": {
        "fp": f"./{ckd_patient_full}", "modifier": "deepsurv",
        "filepaths": paths(ckd_patient_full, ckd_surv_dirs),
    },


}


In [ ]:
preset_modifier = "ckd_surv_event_full"
preset = presets[preset_modifier]
# ==== EDIT ^^ TO SWITCH RUNS ====

fp = preset["fp"]
modifier = preset["modifier"]
filepaths = preset["filepaths"]

print("preset: ", preset_modifier)
print("fp:      ", fp)
print("modifier:", modifier)
print("filepaths:")
for p in filepaths:
    print("  ", p)


In [ ]:
import logging
import sys
import os

modifier = "deepsurv_tte" 
log_path = f'results_logs/{fp}_results_{modifier}_metrics_output.log'

out_folder = "results_files_v2"
out_path =  os.path.join(out_folder, preset_modifier)
print(out_path)
try:
    os.mkdir(out_path)
except FileExistsError:
    pass


In [ ]:
metric_path_1 = os.path.join(out_path, f"{modifier}_metrics_pt1.csv")
print(metric_path_1)
metric_path_2 = os.path.join(out_path, f"{modifier}_metrics_pt2.csv")
print(metric_path_2)
metric_path_3 = os.path.join(out_path, f"{modifier}_metrics_pt3.csv")
print(metric_path_3)

In [ ]:
import pandas as pd
import numpy as np
from sksurv.util import Surv
from sksurv.metrics import cumulative_dynamic_auc, concordance_index_ipcw
import warnings
from tqdm import tqdm
import os
from sklearn.metrics import confusion_matrix
# Suppress all warnings
warnings.filterwarnings("ignore")

def load_and_clean_data(filepath: str) -> pd.DataFrame:
    df = pd.read_csv(filepath)
    return df[["tte_cox_true_time", "tte_cox_true_event", "tte_cox_risk_score"]].dropna()

def create_surv_object(df: pd.DataFrame):
    times = df["tte_cox_true_time"].values
    events = df["tte_cox_true_event"].values.astype(bool)
    surv = Surv.from_arrays(events, times)
    risks = df["tte_cox_risk_score"].values
    return surv, risks

def compute_confusion_matrix(y_true, y_preds):
    tn, fp, fn, tp = confusion_matrix(y_true, y_preds).ravel()
    confusion_m = {
        'True Negative': tn,
        'False Positive': fp,
        'False Negative': fn,
        'True Positive': tp
    }
    return confusion_m


def evaluate_day_365(surv, risks: np.ndarray, threshold: float = 0.2) -> dict:
    eval_day = [365]
    c_index = concordance_index_ipcw(surv, surv, risks)[0]
    _, auc_vals = cumulative_dynamic_auc(surv, surv, risks, eval_day)
    auc_365 = float(auc_vals[0]) if isinstance(auc_vals, (list, np.ndarray)) else float(auc_vals)

    true_events = [int(pair[0] & (pair[1] <= 365)) for pair in surv]#.astype(int)
    # true_events = np.array(true_events, dtype='int')
    # print(true_events)
    predicted_events = (risks > threshold).astype(int)
    # print(predicted_events)

    # confusion matrix
    matrix = compute_confusion_matrix(true_events, predicted_events)
    

    return {"AUC@365": auc_365, "C-index": float(c_index), "Confusion Matrix": matrix}

In [ ]:
# eval pt1
def evaluate_models_pt1(filepaths, verbose=False):
    df_list = []
    print("\n--- Time-dependent AUC and C-index at Day 365 ---")
    for filepath in tqdm(filepaths, desc="Evaluating models at Day 365"):
        name = os.path.splitext(os.path.basename(filepath))[0]
        fn = os.path.basename(filepath)
        df = load_and_clean_data(filepath)
        surv, risks = create_surv_object(df)

        metrics = evaluate_day_365(surv, risks)
        
        row = {"Model": name}
        for k, v in metrics.items():
            
            if isinstance(v, dict):
                for cm_key, cm_val in v.items():
                    # Converts 'True Negative' to 'TN'
                    row[cm_key] = cm_val
            else:
                row[k] = v

        df_list.append(row)

        if verbose:
            print(f"\nModel: {os.path.splitext(fn)[0]}")
            print(f"  AUC@365: {metrics['AUC@365']:.4f}")
            print(f"  C-index: {metrics['C-index']:.4f}")
            print(f"  Confusion Matrix: {metrics['Confusion Matrix']}")
    
    metrics_df = pd.DataFrame(df_list)
    return metrics_df


In [ ]:
# eval pt2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

def evaluate_models_pt2(filepaths, verbose=False):
    model_names = [fp.split("/")[-1].replace("_365DayFuture_detailed_outputs.csv", "") for fp in filepaths]
    eval_times = np.arange(30, 366, 30)

    combined_brier_scores = []

    for fp, model_name in zip(filepaths, model_names):
        df = pd.read_csv(fp)
        df_clean = df.dropna(subset=['tte_cox_true_time', 'tte_cox_true_event', 'cl_prob_1'])

        event_times = df_clean['tte_cox_true_time'].values
        event_observed = df_clean['tte_cox_true_event'].values
        predicted_probs = df_clean['cl_prob_1'].values

        kmf_censor = KaplanMeierFitter()
        kmf_censor.fit(event_times, event_observed == 0)

        for t in eval_times:
            y_true = (event_times > t).astype(int)
            y_pred = 1 - predicted_probs  # survival prob

            G_t = kmf_censor.predict(t)
            weights = (event_times >= t).astype(float) / np.clip(G_t, 1e-5, None)

            brier_score_t = np.mean(weights * (y_pred - y_true) ** 2)
            combined_brier_scores.append({
                'model': model_name,
                'time': t,
                'brier_score': brier_score_t                
            })

    # Convert to DataFrame
    brier_df = pd.DataFrame(combined_brier_scores)

    
    # Plot
    plt.figure(figsize=(10, 6))
    for model_name in brier_df['model'].unique():
        df_plot = brier_df[brier_df['model'] == model_name]
        plt.plot(df_plot['time'], df_plot['brier_score'], marker='o', label=model_name)
    
    title = "Temporal Brier Scores by Model"
    plt.title(title)
    plt.xlabel("Time (days)")
    plt.ylabel("Brier Score")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()

    figname = f"{modifier}_{title}.png"
    figpath = os.path.join(out_path, figname)
    plt.savefig(figpath)
    plt.show()

    return brier_df


In [ ]:
# eval pt3
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lifelines import KaplanMeierFitter
from lifelines.utils import concordance_index
from sksurv.util import Surv
from sksurv.metrics import cumulative_dynamic_auc, brier_score



def sample_shared_test_set(df, horizon, N_PROGRESSOR_SAMPLES, RANDOM_SEED):
    """
    For each patient, sample up to N_PROGRESSOR_SAMPLES distinct failure times
    if they progressed, or one time >= horizon if they did not.
    """
    samples = []
    for pid, group in df.groupby("PatientID"):
        if group['cl_true_label'].iloc[0] == 1:
            # only keep rows where failure actually occurred
            fails = group[group['tte_cox_true_event'] == 1]
            for i in range(min(N_PROGRESSOR_SAMPLES, len(fails))):
                row = fails.sample(n=1, random_state=RANDOM_SEED + i).iloc[0].copy()
                samples.append(row)
        else:
            # sample a censored time that survives at least to the horizon
            cens = group[(group['tte_cox_true_event'] == 0) &
                         (group['tte_cox_true_time'] >= horizon)]
            if cens.empty:
                # if none survive past horizon, skip patient
                continue
            row = cens.sample(n=1, random_state=RANDOM_SEED).iloc[0].copy()
            samples.append(row)
    return pd.DataFrame(samples).drop_duplicates(subset=["PatientID", "tte_cox_true_time"])


def evaluate_model_fullsample_2(df, eval_day, bootstrap=False, random_state=None):
    """
    Compute C-index, time-dependent AUC@eval_day, and IPCW Brier@eval_day
    using the entire test sample without further dropping.
    """
    if bootstrap:
        df = df.sample(n=len(df), replace=True, random_state=random_state)

    df = df.dropna(subset=['tte_cox_true_time', 'tte_cox_true_event', 'tte_cox_risk_score'])
    T = df['tte_cox_true_time'].values
    E = df['tte_cox_true_event'].astype(bool).values
    R = df['tte_cox_risk_score'].values

    # surv = Surv.from_arrays(E, T)
    surv, risks = create_surv_object(df)

    # Concordance
    try:
        # c_index = concordance_index(T, -R, event_observed=E)
        # check
        c_index = concordance_index_ipcw(surv, surv, R)[0]
    except Exception:
        c_index = np.nan

    # Time-dependent AUC
    try:
        _, auc_vals = cumulative_dynamic_auc(surv, surv, risks, eval_day)
        auc_365 = float(auc_vals[0]) if isinstance(auc_vals, (list, np.ndarray)) else float(auc_vals)
    except Exception:
        auc_365 = np.nan

    # IPCW-weighted Brier score
    try:
        # Check if cl_prob_1 exists for Brier score calculation
        if 'cl_prob_1' in df.columns:
            df_brier = df.dropna(subset=['cl_prob_1'])
            event_times = df_brier['tte_cox_true_time'].values
            event_observed = df_brier['tte_cox_true_event'].values
            predicted_probs = df_brier['cl_prob_1'].values
            
            # Fit censoring distribution
            kmf_censor = KaplanMeierFitter()
            kmf_censor.fit(event_times, event_observed == 0)
            
            # Calculate Brier score at eval_day
            y_true = (event_times > eval_day).astype(int)
            y_pred = 1 - predicted_probs  # survival probability
            
            G_t = kmf_censor.predict(eval_day)
            weights = (event_times >= eval_day).astype(float) / np.clip(G_t, 1e-5, None)
            
            brier_365 = np.mean(weights * (y_pred - y_true) ** 2)
    except Exception:
        brier_365 = np.nan

    # return c_index, auc_365, brier_365
    return {"AUC@365": auc_365, "C-index": float(c_index), "brier_score": brier_365}
    


def evaluate_model_fullsample(df, eval_day):
    """
    Compute C-index, time-dependent AUC@eval_day, and IPCW Brier@eval_day
    using the entire test sample without further dropping.
    """
    df = df.dropna(subset=['tte_cox_true_time', 'tte_cox_true_event', 'tte_cox_risk_score'])
    T = df['tte_cox_true_time'].values
    E = df['tte_cox_true_event'].astype(bool).values
    R = df['tte_cox_risk_score'].values
    # print(df.head())

    surv, risks = create_surv_object(df)

    # Concordance
    try:
        c_index = concordance_index_ipcw(surv, surv, R)[0]
    except Exception:
        c_index = np.nan

    # Time-dependent AUC
    try:
        _, auc_vals = cumulative_dynamic_auc(surv, surv, risks, eval_day)
        auc_365 = float(auc_vals[0]) if isinstance(auc_vals, (list, np.ndarray)) else float(auc_vals)
    except Exception:
        auc_365 = np.nan

    # IPCW-weighted Brier score — now uses cl_prob_1, matching pt2
    try:
        if 'cl_prob_1' not in df.columns:
            print(df.columns)
            raise ValueError("cl_prob_1 column not found; cannot compute Brier score consistently with pt2")

        df_brier = df.dropna(subset=['cl_prob_1'])
        surv_brier, _ = create_surv_object(df_brier)
        surv_probs = 1 - df_brier['cl_prob_1'].values  # survival probability, same convention as pt2

        _, brier_vals = brier_score(
            survival_train=surv,
            survival_test=surv_brier,
            estimate=surv_probs,
            times=np.array([eval_day])
        )
        brier_365 = float(brier_vals[0])
        
    except Exception:
        brier_365 = np.nan

    return {"AUC@365": auc_365, "C-index": float(c_index), "brier_score": brier_365}

def plot_risk_distribution(df, model_name):
    plt.hist(df['tte_cox_risk_score'], bins=50, edgecolor='black')
    
    plt.xlabel("Risk Score")
    plt.ylabel("Count")

    title = f"Risk Score Distribution: {model_name}"
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    # figname = f"{modifier}_{title}.png"
    # figpath = os.path.join(out_path, figname)
    # plt.savefig(figpath)
    plt.show()

def plot_risk_vs_tte(df, model_name):
    plt.scatter(
        df['tte_cox_risk_score'],
        df['tte_cox_true_time'],
        c=df['tte_cox_true_event'],
        cmap='coolwarm',
        alpha=0.6
    )
    
    plt.xlabel("Predicted Risk Score")
    plt.ylabel("Time to Event")
    plt.colorbar(label="Event (1) vs Censored (0)")

    title = f"Risk vs Time-to-Event: {model_name}"
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    # figname = f"{modifier}_{title}.png"
    # figpath = os.path.join(out_path, figname)
    # plt.savefig(figpath)
    plt.show()

from concurrent.futures import ProcessPoolExecutor

def bootstrap_once(seed, df, eval_day):
    """
    Single bootstrap iteration: resample with replacement and compute metrics.
    """
    np.random.seed(seed)
    boot_sample = df.sample(n=len(df), replace=True, random_state=seed)
    
    try:
        metrics = evaluate_model_fullsample(boot_sample, eval_day)
        return metrics
    except Exception as e:
        # Return NaN if bootstrap sample fails (e.g., no events)
        return {"AUC@365": np.nan, "C-index": np.nan, "brier_score": np.nan}


def bootstrap_metrics(df, eval_day, n_iterations=1000, n_workers=8):

    print(f"Bootstrapping with {n_iterations} iterations using {n_workers} workers...")
    
    # Generate random seeds for reproducibility
    seeds = np.random.randint(0, 100000, size=n_iterations)
    
    # Initialize storage for bootstrap results
    boot_metrics = {
        'C-index': [],
        f'AUC@{eval_day}': [],
        f'Brier@{eval_day}': []
    }
    
    # Parallel bootstrap
    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = [executor.submit(bootstrap_once, s, df, eval_day) for s in seeds]
        
        for i, f in enumerate(futures):
            if (i + 1) % 100 == 0:
                print(f"  Completed {i + 1}/{n_iterations} iterations...")
            
            result = f.result()
            boot_metrics['C-index'].append(result['C-index'])
            boot_metrics[f'AUC@{eval_day}'].append(result[f'AUC@{eval_day}'])
            boot_metrics[f'Brier@{eval_day}'].append(result['brier_score'])
    
    print("Bootstrapping completed.")


    summary = {k: (np.mean(v), np.percentile(v, 2.5), np.percentile(v, 97.5)) 
    for k, v in boot_metrics.items()}
    
    return summary

def evaluate_models_pt3(filepaths, EVAL_DAY=365, n_boot=1000, n_workers=8, 
                        N_PROGRESSOR_SAMPLES=5, RANDOM_SEED=42, verbose=False):
    
    np.random.seed(RANDOM_SEED)
    random.seed(RANDOM_SEED)
    
    # Creates a shared test set so all models are evaluated on the same patient-timepoint pairs
    base_df = pd.read_csv(filepaths[0])
    test_df = sample_shared_test_set(base_df, EVAL_DAY, N_PROGRESSOR_SAMPLES, RANDOM_SEED)

    p_count = (test_df['tte_cox_true_event'] == 1).sum()
    tot = len(test_df)
    prev = 100 * p_count / tot if tot > 0 else 0
    print(f"Shared-test prevalence: {prev:.1f}% ({p_count}/{tot})\n")

    results = {}
    all_boot_metrics = {}
    df_list = []
    
    for filepath in filepaths:
        name = os.path.splitext(os.path.basename(filepath))[0]
        print(f"\nProcessing file: {name}")

        df_full = pd.read_csv(filepath)
        
        print()
        print("Full Data shape:", df_full.shape)
        #  MERGE WITH SHARED TEST SET 
        # use the same timepoints across all models
        print(test_df.columns)
        print(df_full.columns)

        merged = (
            test_df[['PatientID', 'tte_cox_true_time', 'tte_cox_true_event']]
            .merge(
                df_full[['PatientID', 'tte_cox_true_time', 'tte_cox_risk_score', 'cl_prob_1']],
                on=['PatientID', 'tte_cox_true_time'],
                how='inner'
            )
        )
        print(merged.columns)
        df_full = merged

        print("Data shape:", merged.shape)
        print(merged.head())

        print("Data shape:", df_full.shape)
        print(df_full.head())
        
        print(f"Computing metrics at evaluation day {EVAL_DAY}...")
        metrics = evaluate_model_fullsample(df_full, EVAL_DAY)

        
        print("Metrics: ")
        for key, value in metrics.items():
            print(f"  {key}: {value:.4f}")
        
        boot_metrics = bootstrap_metrics(df_full, EVAL_DAY, n_iterations=n_boot, n_workers=n_workers)
        
        if verbose:
            print("Bootstrapped Metrics with 95% Confidence Intervals:")
            for k, (mean, low, high) in boot_metrics.items():
                print(f"  {k}: {mean:.4f} [{low:.4f}, {high:.4f}]")
        
        # Build output row
        row = {"Model": name, "Eval_Day": EVAL_DAY}
        
        for k, v in metrics.items():
            if isinstance(v, dict):
                for cm_key, cm_val in v.items():
                    row[cm_key] = cm_val
            else:
                row[k] = v
        
        for k, (mean, low, high) in boot_metrics.items():
            row[f"{k}_boot_mean"] = mean
            row[f"{k}_ci_lower"] = low
            row[f"{k}_ci_upper"] = high
        
        df_list.append(row)
        
        results[name] = {
            "merged_data": df_full,
            "eval_day": EVAL_DAY,
            "metrics": metrics
        }
        all_boot_metrics[name] = boot_metrics
        
        plot_risk_distribution(df_full, name)
        plot_risk_vs_tte(df_full, name)
    
    metrics_df = pd.DataFrame(df_list)
    return metrics_df#, results, all_boot_metrics




In [ ]:
from contextlib import redirect_stdout, redirect_stderr

with open(log_path, 'w') as f:
    with redirect_stdout(f), redirect_stderr(f):
        
        metrics_1 = evaluate_models_pt1(filepaths, verbose=False)
        metrics_2 = evaluate_models_pt2(filepaths, verbose=False)
        metrics_3 = evaluate_models_pt3(filepaths, EVAL_DAY = 365, n_boot= 1000, n_workers=20
        , N_PROGRESSOR_SAMPLES= 5, RANDOM_SEED = 42, verbose=False)



In [ ]:
def metrics_to_csv(path, metrics):
    print(path)
    try:
        # Attempt Polars syntax first
        metrics.write_csv(path)
    except AttributeError:
        # Fallback to Pandas syntax
        metrics.to_csv(path, index=False)

    return

In [ ]:
metrics_to_csv(metric_path_1, metrics_1)

In [ ]:
metrics_to_csv(metric_path_2, metrics_2)

In [ ]:
metrics_to_csv(metric_path_3, metrics_3)

In [ ]:
metrics_1

In [ ]:
metrics_2.head()

In [ ]:
metrics_3